# Day 9 — File I/O
### Python for Data Science · Module 1 · Topic 1.9

**Prepared & presented by Srinivasa Sai Chava**  ·  Boston University

---

**Session length:** 2 hours
**Format:** 90 min concepts + live coding · 30 min practice

| # | What we cover | Time |
|---|---|---|
| 1 | Reading text files: `open()`, `with`, four ways to read | 25 min |
| 2 | Writing and file modes | 20 min |
| 3 | CSV files | 25 min |
| 4 | Paths and errors | 10 min |
| 5 | Mini build: CSV in, report out | 10 min |
| 6 | **Practice notebook (separate file)** | 30 min |

> **Two habits to leave with:** always use `with`, and never open a real file in `"w"` mode
> by accident.
>
> **This is the bridge to the rest of the course.** In two weeks `pd.read_csv("data.csv")`
> gives you a DataFrame in one line. Today you do the same job by hand — and that is the
> point. When a file has an odd separator or a strange encoding, pandas will not save you
> unless you understand what it was doing.

⚠️ **In Colab, files you create disappear when the session ends.** Everything below is safe
to run: this notebook creates its own sample files in the current folder.

---
## 0. Set up some sample files

Run this first — everything afterwards uses these.

In [3]:
# Create a small text file
with open("names.txt", "w", encoding="utf-8") as f:
    f.write("Ravi\n")
    f.write("Sara\n")
    f.write("Amit")          # note: no newline on the last one

# Create a CSV, including a value with a comma inside it
import csv
with open("marks.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["name", "city", "subject", "mark"])
    w.writerows([
        ["Ravi", "Pune",       "python", 88],
        ["Sara", "Mumbai, MH", "python", 91],
        ["Ravi", "Pune",       "stats",  71],
        ["Amit", "Delhi",      "stats",  64],
    ])

print("Sample files created.")
print(open("marks.csv", encoding="utf-8").read())

Sample files created.
name,city,subject,mark
Ravi,Pune,python,88
Sara,"Mumbai, MH",python,91
Ravi,Pune,stats,71
Amit,Delhi,stats,64



---
# 1. Reading text files

## 1.1 `open()` and why you always use `with`

In [4]:
# THE MANUAL WAY - you must remember to close
f = open("names.txt", encoding="utf-8")
text = f.read()
f.close()                      # if read() had raised, this never runs
print(text)
print("closed?", f.closed)

Ravi
Sara
Amit
closed? True


In [5]:
# THE WAY YOU SHOULD ALWAYS WRITE IT
with open("names.txt", encoding="utf-8") as f:
    text = f.read()

print(text)
print("closed?", f.closed)     # True - automatically

Ravi
Sara
Amit
closed? True


In [6]:
# The guarantee: the file closes even when the block raises
try:
    with open("names.txt", encoding="utf-8") as f:
        raise ValueError("something went wrong")
except ValueError:
    pass

print("closed even after an error?", f.closed)

closed even after an error? True


An open file holds an operating-system handle. Leak enough of them and the program cannot
open anything at all.

**Always pass `encoding="utf-8"`.** Without it Python uses whatever the machine prefers, so
a file that reads perfectly on your laptop can fail on someone else's. One argument removes
an entire category of bug.

> `with` is a general Python feature, not a file feature — you will meet it again with
> database connections and locks.

## 1.2 Four ways to read

| Method | Gives you | Use it when |
|---|---|---|
| `f.read()` | the whole file as ONE string | the file is small and you want it all |
| `f.readline()` | the next single line | you need just the header, or one line |
| `f.readlines()` | a list of every line | you need a list and the file is small |
| `for line in f:` | one line at a time, lazily | **almost always** — any size of file |

In [7]:
with open("names.txt", encoding="utf-8") as f:
    print("read():", repr(f.read()))

with open("names.txt", encoding="utf-8") as f:
    print("readline():", repr(f.readline()))

with open("names.txt", encoding="utf-8") as f:
    print("readlines():", f.readlines())

read(): 'Ravi\nSara\nAmit'
readline(): 'Ravi\n'
readlines(): ['Ravi\n', 'Sara\n', 'Amit']


In [8]:
# THE ONE TO DEFAULT TO
with open("names.txt", encoding="utf-8") as f:
    for line in f:
        print(repr(line))

# Reads one line at a time - a 10 GB file uses no more memory than a small one.

'Ravi\n'
'Sara\n'
'Amit'


### ⚠️ Every line keeps its newline

In [9]:
with open("names.txt", encoding="utf-8") as f:
    for line in f:
        print(f"{repr(line):12} == 'Ravi'? {line == 'Ravi'}   "
              f"stripped == 'Ravi'? {line.strip() == 'Ravi'}")

# Note the LAST line has no \n at all - that asymmetry causes real bugs.

'Ravi\n'     == 'Ravi'? False   stripped == 'Ravi'? True
'Sara\n'     == 'Ravi'? False   stripped == 'Ravi'? False
'Amit'       == 'Ravi'? False   stripped == 'Ravi'? False


### ⚠️ A file is exhausted after one pass

In [10]:
with open("names.txt", encoding="utf-8") as f:
    first  = f.read()
    second = f.read()      # already at the end

print("first :", repr(first[:10]), "...")
print("second:", repr(second))      # '' - empty!

# Same idea as Day 6's generator exhaustion. Store what you read,
# or open the file again.

first : 'Ravi\nSara\n' ...
second: ''


---
# 2. Writing and file modes

| Mode | Means | If the file exists | If it does not |
|---|---|---|---|
| `"r"` | read (the default) | reads it | `FileNotFoundError` |
| `"w"` | write | **ERASES IT COMPLETELY** | creates it |
| `"a"` | append | adds to the end | creates it |
| `"x"` | exclusive create | `FileExistsError` | creates it |
| `"r+"` | read and write | opens without erasing | `FileNotFoundError` |

> ### ⚠️ `"w"` does not warn you
> There is no undo, no recycle bin, no confirmation. Opening a real data file in `"w"` mode
> destroys it instantly.
>
> **Protect yourself:** use `"a"` to add, use `"x"` when the file must not already exist,
> and write results to a **new** filename rather than over your input.

In [11]:
# Demonstrating "w" - watch the file get replaced
with open("demo.txt", "w", encoding="utf-8") as f:
    f.write("original content\n")
print("after first write :", repr(open("demo.txt", encoding="utf-8").read()))

with open("demo.txt", "w", encoding="utf-8") as f:   # "w" again
    f.write("new\n")
print("after second write:", repr(open("demo.txt", encoding="utf-8").read()))
#   The original line is gone.

after first write : 'original content\n'
after second write: 'new\n'


In [12]:
# "a" keeps what is already there
with open("demo.txt", "a", encoding="utf-8") as f:
    f.write("appended\n")

print(repr(open("demo.txt", encoding="utf-8").read()))

'new\nappended\n'


In [13]:
# "x" refuses rather than overwriting
try:
    with open("demo.txt", "x", encoding="utf-8") as f:
        f.write("this will not happen")
except FileExistsError as e:
    print("FileExistsError:", e)

FileExistsError: [Errno 17] File exists: 'demo.txt'


## 2.1 `write()` adds no newline

In [14]:
with open("out.txt", "w", encoding="utf-8") as f:
    f.write("Ravi")
    f.write("Sara")

print(repr(open("out.txt", encoding="utf-8").read()))    # 'RaviSara'

'RaviSara'


In [28]:
names = ["Ravi", "Sara", "Amit"]

# Three ways to write many lines
with open("out.txt", "w", encoding="utf-8") as f:
    for n in names:
        f.write(n + "\n")                  # you add the newline

with open("out2.txt", "w", encoding="utf-8") as f:
    f.writelines(n + "\n" for n in names)  # writelines adds NO newlines either

with open("out3.txt", "w", encoding="utf-8") as f:
    for n in names:
        print(n, file=f)                    # print DOES add one

for path in ["out.txt", "out2.txt", "out3.txt"]:
    print(path, "->", repr(open(path, encoding="utf-8").read()))

out.txt -> 'Ravi\nSara\nAmit\n'
out2.txt -> 'Ravi\nSara\nAmit\n'
out3.txt -> 'Ravi\nSara\nAmit\n'


Two details worth knowing: `write()` returns the number of characters written, which is
occasionally useful and usually ignored. And nothing is guaranteed to reach the disk until
the file is closed — which the `with` block handles for you.

---
# 3. CSV files

## 3.1 Why `line.split(",")` is not enough

In [32]:
# Our sample file has a value containing a comma: "Mumbai, MH"
with open("marks.csv", encoding="utf-8") as f:
    for line in f:
        print(line.strip().split(","))

# Look at Sara's row - split() produced FIVE fields instead of four.

['name', 'city', 'subject', 'mark']
['Ravi', 'Pune', 'python', '88']
['Sara', '"Mumbai', ' MH"', 'python', '91']
['Ravi', 'Pune', 'stats', '71']
['Amit', 'Delhi', 'stats', '64']


In [38]:
import csv

with open("marks.csv", encoding="utf-8") as f:
    for row in csv.reader(f):
        print(row)

# The csv module understands quoted fields. Sara's city stays in one piece.

['Ravi', 'Pune', 'python', '88']
['name', 'city', 'subject', 'mark']
['Sara', 'Mumbai, MH', 'python', '91']
['Ravi', 'Pune', 'stats', '71']
['Amit', 'Delhi', 'stats', '64']


### Two things to remember when opening a CSV

```python
open(path, newline="", encoding="utf-8")
```

- `newline=""` lets the csv module handle line endings itself. Without it you can get blank
  rows between every record on Windows.
- **Every value comes back as a string.** `"88"` is text until you cast it — Day 1's
  `input()` lesson in a new place.

## 3.2 `csv.reader` vs `csv.DictReader`

In [41]:
import csv

# reader - each row is a LIST, addressed by POSITION
with open("marks.csv", newline="", encoding="utf-8") as f:
    reader = csv.reader(f)
    header = next(reader)              # skip the header row
    print("header:", header)
    for row in reader:
        print(row[0])     # name, mark

header: ['Ravi', 'Pune', 'python', '88']
name
Sara
Ravi
Amit


In [44]:
# DictReader - each row is a DICT, addressed by NAME
with open("marks.csv", newline="", encoding="utf-8") as f:
    for row in csv.DictReader(f):
        print(row["name"],row["city"], int(row["mark"]))

# The header becomes the keys automatically - no next() needed.

Ravi Pune 88
Sara Mumbai, MH 91
Ravi Pune 71
Amit Delhi 64


In [43]:
# What a DictReader row actually looks like
with open("marks.csv", newline="", encoding="utf-8") as f:
    first = next(csv.DictReader(f))
    second = next(csv.DictReader(f))
print(first)
print(second)
print(type(first).__name__)

{'name': 'Ravi', 'city': 'Pune', 'subject': 'python', 'mark': '88'}
{'Sara': 'Ravi', 'Mumbai, MH': 'Pune', 'python': 'stats', '91': '71'}
dict


> ### Prefer `DictReader`
>
> `row["mark"]` survives someone adding a column; `row[3]` silently starts reading the wrong
> field. It is Day 5's argument for dictionaries over lists — the data explains itself —
> applied to files.

## 3.3 Writing a CSV

In [46]:
import csv

rows = [["Ravi", 88], ["Sara", 91],["Sai", 80]]

with open("simple.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f) 
    w.writerow(["name", "mark"])     # header
    w.writerows(rows)                # all the rest at once

print(open("simple.csv", encoding="utf-8").read())

name,mark
Ravi,88
Sara,91
Sai,80



In [22]:
# DictWriter - fieldnames decides both the header and the column order
rows = [{"name": "Ravi", "mark": 88},
        {"name": "Sara", "mark": 91}]

with open("dict.csv", "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["name", "mark"])
    w.writeheader()
    w.writerows(rows)

print(open("dict.csv", encoding="utf-8").read())

name,mark
Ravi,88
Sara,91



In [50]:
# The csv module quotes for you when a value contains a comma
with open("quoted.csv", "w", newline="", encoding="utf-8") as f:
    csv.writer(f).writerow(["Sara", "Mumbai, MH"])
    csv.writer(f).writerow(["Ravi", "Mumbai, MA"])

print(repr(open("quoted.csv", encoding="utf-8").read()))

# Building CSV by hand with f-strings works right up until a value contains
# a comma - and then it quietly produces a broken file.

'Sara,"Mumbai, MH"\nRavi,"Mumbai, MA"\n'


---
# 4. Paths and errors

| Error | Cause |
|---|---|
| `FileNotFoundError` | wrong name or folder |
| `PermissionError` | no rights to open it |
| `IsADirectoryError` | that path is a folder |
| `UnicodeDecodeError` | wrong encoding |
| `FileExistsError` | `"x"` mode, file already there |

In [24]:
try:
    with open("does_not_exist.csv", encoding="utf-8") as f:
        rows = f.readlines()
except FileNotFoundError:
    print("No data file — starting empty")
    rows = []

print("rows:", rows)

No data file — starting empty
rows: []


In [25]:
from pathlib import Path

print("exists?", Path("marks.csv").exists())
print("exists?", Path("nope.csv").exists())

# Useful, but try/except is safer: the file could vanish between
# the check and the open.

exists? True
exists? False


A bare `"data.csv"` means *"in the current folder"*. In Colab that folder empties when the
session ends, so anything you write today is temporary — upload with the file panel on the
left, or mount Google Drive to keep things.

---
# 5. Putting it together — CSV in, report out

In [26]:
import csv
from collections import defaultdict


def load_marks(path):
    """Read the CSV into a list of clean dictionaries."""
    rows = []
    with open(path, newline="", encoding="utf-8") as f:
        for row in csv.DictReader(f):
            row["mark"] = int(row["mark"])       # cast at the door
            rows.append(row)
    return rows


def summarise(rows):
    """Average mark per subject."""
    totals = defaultdict(list)
    for r in rows:
        totals[r["subject"]].append(r["mark"])
    return {s: sum(m) / len(m) for s, m in totals.items()}


def write_report(summary, path):
    """Write the summary to a new CSV."""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["subject", "average"])
        w.writerows([[s, f"{a:.1f}"] for s, a in summary.items()])


rows = load_marks("marks.csv")
print("loaded:", len(rows), "rows")
print("comma survived:", rows[1]["city"])
print("summary:", summarise(rows))

write_report(summarise(rows), "report.csv")      # a NEW filename
print()
print(open("report.csv", encoding="utf-8").read())

loaded: 4 rows
comma survived: Mumbai, MH
summary: {'python': 89.5, 'stats': 67.5}

subject,average
python,89.5
stats,67.5



What each piece contributes:

- **`DictReader`** — the header becomes the keys
- **Cast on load** — `"88"` becomes `88` the moment it enters the program
- **`with`, twice** — both files close themselves
- **Day 5's grouping** — `defaultdict` collects marks per subject
- **Day 6's comprehension** — builds the summary dictionary
- **One job per function** — load · summarise · write
- **A new output file** — the input is never touched

In two weeks pandas does all of this in three lines — and you will know what it is doing.

---
# 6. Recap — the twelve things to remember

1. Always use `with open(...) as f:` — it closes the file for you.
2. Pass `encoding="utf-8"` on every text file.
3. `"r"` reads · `"w"` **ERASES** · `"a"` appends · `"x"` refuses to overwrite.
4. `for line in f:` is the default way to read — any file size.
5. Every line keeps its `\n`. Use `.strip()` before comparing.
6. `f.write()` adds no newline. `print(..., file=f)` does.
7. `line.split(",")` breaks on commas inside quoted fields.
8. Use `csv.reader` for lists, `csv.DictReader` for dicts.
9. Open CSVs with `newline=""` to avoid blank rows.
10. Every CSV value arrives as a **string**. Cast it at the door.
11. Wrap opens in `try / except FileNotFoundError`.
12. Write results to a **new** file — never over your input.

---

### 📝 Now open **`Day9_Practice_Questions.ipynb`** for the 30-minute practice session.

### Homework
- Write a word-frequency counter that reads a text file.
- Read a CSV and write out only the rows that pass a filter.
- Add `try`/`except` to every file function you have written.

### Next class — Topic 1.10: Exception Handling & Modules
`try` / `except` / `finally`, raising and writing your own exceptions, importing modules
and packages.

---
*Slides & notebooks by Srinivasa Sai Chava · Boston University*